# Notebook 05 — Neighbourhood Clustering

**Member 3 task A** — Segment every (city, neighbourhood) row in `neighbourhood_kpis.csv` into interpretable STR-pressure clusters.

Outputs:
- `cluster_label` and `cluster_distance` columns appended to the knowledge layer
- Silhouette score in `reports/model_metrics.md`
- Cluster scatter plots and maps in `reports/figures/clustering/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

from src.data_io import PROCESSED_DIR

RANDOM_STATE = 42
FIGURES_DIR = Path('..') / 'reports' / 'figures' / 'clustering'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

## 1. Load neighbourhood KPIs

In [ ]:
kpis = pd.read_csv(PROCESSED_DIR / 'neighbourhood_kpis.csv')
print(f"Shape: {kpis.shape}")
print(f"Cities: {kpis['city'].value_counts().to_dict()}")
kpis.head(3)

## 2. Feature selection & scaling

In [ ]:
CLUSTER_FEATURES = [
    'str_density',
    'entire_home_share',
    'commercial_host_share',
    'multi_listing_host_share',
    'avg_occupancy',
    'breach_rate_90',
]

X = kpis[CLUSTER_FEATURES].fillna(0).copy()

# MinMax scaling: preserves relative magnitudes within each feature (0–1 range)
# Preferred over StandardScaler here because the features are bounded ratios/counts
# and outlier sensitivity would distort cluster geometry.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Feature matrix shape:", X_scaled.shape)
print("Any NaN after fillna:", np.isnan(X_scaled).any())

## 3. Elbow + silhouette sweep (k = 2 – 6)

In [ ]:
inertias, sils = [], []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))
    print(f"k={k}  inertia={km.inertia_:,.0f}  silhouette={sils[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(list(k_range), inertias, marker='o', color='steelblue')
ax1.set_xlabel('k')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow curve')
ax1.axvline(3, color='red', linestyle='--', alpha=0.6, label='k=3 chosen')
ax1.legend()

ax2.plot(list(k_range), sils, marker='o', color='seagreen')
ax2.set_xlabel('k')
ax2.set_ylabel('Silhouette score')
ax2.set_title('Silhouette vs k')
ax2.axvline(3, color='red', linestyle='--', alpha=0.6, label='k=3 chosen')
ax2.legend()

plt.suptitle('KMeans cluster selection — neighbourhood KPIs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved elbow_silhouette.png")

## 4. Fit final model — k = 3

In [ ]:
km_final = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
raw_labels = km_final.fit_predict(X_scaled)
kpis['_raw_cluster'] = raw_labels

# Silhouette score & per-sample distances
final_sil = silhouette_score(X_scaled, raw_labels)
distances = km_final.transform(X_scaled)  # shape (n, 3)
# Distance to the assigned centroid
kpis['cluster_distance'] = distances[np.arange(len(kpis)), raw_labels].round(5)

print(f"Final silhouette score (k=3): {final_sil:.4f}")

# Inspect centroids to assign interpretable labels
centroid_df = pd.DataFrame(
    scaler.inverse_transform(km_final.cluster_centers_),
    columns=CLUSTER_FEATURES
)
centroid_df.index.name = 'cluster_id'
print("\nCluster centroids (original scale):")
print(centroid_df.round(3).to_string())

## 5. Assign interpretable labels

In [ ]:
# Label assignment rationale (document in model_metrics.md):
# ─────────────────────────────────────────────────────────────────────────────
# Cluster with HIGHEST str_density + HIGHEST entire_home_share          → saturated
#   These subdivisions have the densest STR footprint and the largest share of
#   entire homes — the clearest proxy for long-term housing displacement.
#
# Cluster with HIGHEST breach_rate_90 + HIGH occupancy                  → emerging
#   High regulatory breach rate signals intense commercial activity even if the
#   raw listing count is lower. These areas are enforcement priority hotspots.
#
# Cluster with LOW values across all features                            → low_impact
#   Sparse STR presence; minimal policy concern under current data.
# ─────────────────────────────────────────────────────────────────────────────

cluster_stats = kpis.groupby('_raw_cluster')[CLUSTER_FEATURES].mean()
print(cluster_stats.round(3).to_string())

# Determine mapping from raw cluster id → label
# Use density to identify saturated; breach_rate for emerging
density_rank = cluster_stats['str_density'].rank(ascending=False)
breach_rank  = cluster_stats['breach_rate_90'].rank(ascending=False)

saturated_id = int(density_rank.idxmin())  # lowest rank = highest density
remaining = [c for c in [0,1,2] if c != saturated_id]
emerging_id  = int(breach_rank[remaining].idxmin())  # highest breach among remaining
low_impact_id = [c for c in remaining if c != emerging_id][0]

LABEL_MAP = {saturated_id: 'saturated', emerging_id: 'emerging', low_impact_id: 'low_impact'}
kpis['cluster_label'] = kpis['_raw_cluster'].map(LABEL_MAP)

print("\nLabel mapping:", LABEL_MAP)
print("\nCluster counts:")
print(kpis['cluster_label'].value_counts())
print("\nMean KPIs by cluster_label:")
print(kpis.groupby('cluster_label')[CLUSTER_FEATURES].mean().round(3).to_string())

## 6. Validation — cluster label stability check

In [ ]:
# Confirm reproducibility: re-fit with same random_state, labels must match
km_check = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
check_labels = km_check.fit_predict(X_scaled)
assert (check_labels == raw_labels).all(), "Labels not stable across reruns!"
print("✓ Cluster labels are stable across reruns (random_state=42)")
print(f"✓ Silhouette score: {final_sil:.4f}")

## 7. Silhouette plot per cluster

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
y_lower = 10
colours = {'saturated': '#d62728', 'emerging': '#ff7f0e', 'low_impact': '#2ca02c'}

for label in ['saturated', 'emerging', 'low_impact']:
    mask = kpis['cluster_label'] == label
    sil_vals = silhouette_samples(X_scaled, raw_labels)[mask.values]
    sil_vals.sort()
    n = sil_vals.shape[0]
    y_upper = y_lower + n
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_vals, alpha=0.7, color=colours[label], label=label)
    ax.text(-0.05, y_lower + 0.5 * n, label, fontsize=9)
    y_lower = y_upper + 10

ax.axvline(final_sil, color='black', linestyle='--', label=f'Mean silhouette = {final_sil:.3f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette plot — neighbourhood clusters (k=3)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved silhouette_plot.png")

## 8. Cluster scatter plots

In [ ]:
colour_map = {'saturated': '#d62728', 'emerging': '#ff7f0e', 'low_impact': '#2ca02c'}
PAIR_AXES = [
    ('str_density', 'entire_home_share'),
    ('breach_rate_90', 'avg_occupancy'),
    ('commercial_host_share', 'breach_rate_90'),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (xc, yc) in zip(axes, PAIR_AXES):
    for label, grp in kpis.groupby('cluster_label'):
        ax.scatter(grp[xc], grp[yc], c=colour_map[label], label=label, alpha=0.65, s=18)
    ax.set_xlabel(xc)
    ax.set_ylabel(yc)
    ax.set_title(f'{xc} vs {yc}')

handles = [mpatches.Patch(color=v, label=k) for k, v in colour_map.items()]
fig.legend(handles=handles, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Cluster scatter plots — neighbourhood STR pressure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved cluster_scatter.png")

## 9. Cluster composition by city

In [ ]:
city_cluster = kpis.groupby(['city', 'cluster_label']).size().unstack(fill_value=0)
print("Cluster distribution by city:")
print(city_cluster)

city_cluster.plot(kind='bar', color=[colour_map[c] for c in city_cluster.columns],
                  figsize=(8, 4), edgecolor='white')
plt.title('Cluster composition by city')
plt.ylabel('Number of subdivisions')
plt.xlabel('City')
plt.xticks(rotation=0)
plt.legend(title='Cluster', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cluster_by_city.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved cluster_by_city.png")

## 10. Save the enriched knowledge layer (interim)

In [ ]:
# Drop helper column
kpis = kpis.drop(columns=['_raw_cluster'])

# Save as knowledge_layer_interim.csv (Member 3 adds risk score next)
out_path = PROCESSED_DIR / 'knowledge_layer_interim.csv'
kpis.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Shape: {kpis.shape}")
print("Columns:", list(kpis.columns))

## 11. Write silhouette score to model_metrics.md

In [ ]:
metrics_path = Path('..') / 'reports' / 'model_metrics.md'
sil_block = f"""
## Clustering — KMeans (k=3)

| Metric | Value |
|---|---|
| Algorithm | KMeans |
| k | 3 |
| random_state | 42 |
| Scaling | MinMaxScaler |
| Silhouette score | {final_sil:.4f} |

### Cluster features

Clustering was run on: `{', '.join(CLUSTER_FEATURES)}`

### Cluster centroids (original scale)

{centroid_df.round(3).to_markdown()}

### Cluster label rationale

| Label | Distinguishing signal |
|---|---|
| `saturated` | Highest STR density AND highest entire-home share — strongest housing displacement signal |
| `emerging` | Highest 90-night breach rate AND high occupancy — intense commercial activity, enforcement priority |
| `low_impact` | Low values across all features — sparse STR presence, minimal policy concern |

### Cluster sizes

{kpis['cluster_label'].value_counts().to_frame().to_markdown()}

"""

# Append (create if not exists)
with open(metrics_path, 'a') as f:
    f.write(sil_block)
print(f"Appended clustering metrics to {metrics_path}")

In [ ]:
print("\n✅ Notebook 05 complete.")
print(f"   Silhouette score: {final_sil:.4f}")
print(f"   Cluster counts: {kpis['cluster_label'].value_counts().to_dict()}")
print(f"   Output: data/processed/knowledge_layer_interim.csv")
print(f"   Figures: reports/figures/clustering/")